In [1]:
# Principal:    report_user
# Role:         report_viewer
# Catalog Role: report_catalog_role
# Привилегии:   TABLE_READ_DATA (table: gold.mart_top_customers)
#
# Матрица доступов:
#   bronze: НЕТ ДОСТУПА
#   silver: НЕТ ДОСТУПА
#   gold:   только gold.mart_top_customers (точечный доступ на уровне таблицы)
#
# FORBIDDEN: SELECT gold.mart_sales_by_category, SELECT silver.*, SELECT bronze.*,
#            INSERT/CREATE в любом namespace

In [2]:
import os
from pyspark.sql import SparkSession

client_id = os.environ["REPORT_USER_CLIENT_ID"]
client_secret = os.environ["REPORT_USER_CLIENT_SECRET"]
credential = f"{client_id}:{client_secret}"

spark = (
    SparkSession.builder
    .appName("lakehouse-report-user")
    .config("spark.sql.catalog.lakehouse.credential", credential)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [3]:
print("[ALLOWED] TABLE_READ_DATA — lakehouse.gold.mart_top_customers")
spark.sql("""
    SELECT
        customer_id,
        name,
        recency_days,
        frequency,
        monetary,
        segment
    FROM lakehouse.gold.mart_top_customers
    LIMIT 5
""").show(truncate=False)
print("SELECT ok")

[ALLOWED] TABLE_READ_DATA — lakehouse.gold.mart_top_customers
+-----------+------------------------------+------------+---------+--------+-------+
|customer_id|name                          |recency_days|frequency|monetary|segment|
+-----------+------------------------------+------------+---------+--------+-------+
|69         |Зайцев Ипполит Харлампович    |20          |15       |64430.24|High   |
|201        |Маргарита Юрьевна Полякова    |13          |10       |49241.54|High   |
|35         |Сорокин Демьян Фадеевич       |17          |15       |46409.88|High   |
|185        |Евдокимов Якуб Тарасович      |27          |15       |46384.48|High   |
|177        |Прасковья Робертовна Меркушева|45          |13       |46205.17|High   |
+-----------+------------------------------+------------+---------+--------+-------+

SELECT ok


In [4]:
print("[FORBIDDEN] SELECT gold.mart_sales_by_category (другая таблица в том же namespace)")
try:
    spark.sql("""
        SELECT
            category_name,
            total_revenue
        FROM lakehouse.gold.mart_sales_by_category
        LIMIT 1
    """).show(truncate=False)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

print("[FORBIDDEN] SELECT silver.customers")
try:
    spark.sql("""
        SELECT
            id,
            name
        FROM lakehouse.silver.customers
        LIMIT 1
    """).show(truncate=False)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

print("[FORBIDDEN] SELECT bronze.raw_orders")
try:
    spark.sql("""
        SELECT
            id,
            status
        FROM lakehouse.bronze.raw_orders
        LIMIT 1
    """).show(truncate=False)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

print("[FORBIDDEN] INSERT в gold.mart_top_customers")
try:
    spark.sql("""
        INSERT INTO lakehouse.gold.mart_top_customers
        VALUES (99999, 'test', 1, CAST(1 AS BIGINT), CAST(100.00 AS DECIMAL(14,2)), 'Low')
    """)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

[FORBIDDEN] SELECT gold.mart_sales_by_category (другая таблица в том же namespace)
[ОЖИДАЕМО] Доступ запрещён: An error occurred while calling o43.sql.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'report_user' with activated PrincipalRoles '[report_viewer]' and activated grants via '[report_viewer, report_catalog_role]' is not authorized for op LOAD_TABLE
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)

[FORBIDDEN] SELECT silver.customers
[ОЖИДАЕМО] Доступ запрещён: An error occurred while calling o43.sql.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'report_user' with activated PrincipalRoles '[report_viewer]' and activated grants via '[report_viewer, report_catalog_role]' is not authorized for op LOAD_TABLE
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)

[FORBIDDEN] SELECT bronze.raw_orders
[ОЖИДАЕМО] Доступ запрещён: An error occurred wh